# Tool Use with Claude: Reminder Project

Stand-in for the "pre-built" code the *Tool use with Claude* lessons reference (lessons 21-28 and onward).

**Run order:** Run All after any kernel restart. Sections 1-5 define everything; section 6 has test conversations.

How it fits together:
- **Tool functions** (section 3) are plain Python that does the real work.
- **Schemas** (section 4) describe each tool to Claude.
- **`run_tool`** routes a tool name to its function; **`run_tools`** handles every tool-use block in a reply.
- **`run_conversation`** loops: call Claude, run any requested tools, send results back, repeat until Claude stops asking for tools.

## 1. Setup

In [ ]:
import json
from datetime import datetime, timedelta

from anthropic import Anthropic
from anthropic.types import ToolParam
from dotenv import load_dotenv

load_dotenv()          # reads ANTHROPIC_API_KEY from .env
client = Anthropic()
MODEL = "claude-haiku-4-5"

## 2. Chat helpers (tool-aware versions)

In the tool lessons the helpers change slightly:
- Messages can hold **content blocks** (text, tool_use, tool_result), not just strings.
- `chat` returns the **full message**, because you need to inspect its blocks and `stop_reason`.
- `text_from_message` pulls out just the text when you want it.

In [ ]:
def add_user_message(messages, content):
    """content can be a string or a list of content blocks (e.g. tool results)."""
    messages.append({"role": "user", "content": content})


def add_assistant_message(messages, message):
    """Accepts a full Message from the API, a list of blocks, or a string."""
    content = message.content if hasattr(message, "content") else message
    messages.append({"role": "assistant", "content": content})


def chat(messages, system=None, temperature=1.0, stop_sequences=None, tools=None):
    """Send the conversation to Claude and return the full Message object."""
    params = {
        "model": MODEL,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    if tools:
        params["tools"] = tools
    return client.messages.create(**params)


def text_from_message(message):
    """Join all text blocks in a Message into one string."""
    return "\n".join(block.text for block in message.content if block.type == "text")

## 3. Tool functions

In [ ]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


def add_duration_to_datetime(datetime_str, duration=0, unit="days",
                             input_format="%Y-%m-%d"):
    """Add a duration to a date/time string. Supports seconds through years.
    Months and years respect month lengths (Jan 31 + 1 month -> Feb 28/29)."""
    date = datetime.strptime(datetime_str, input_format)
    unit = unit.lower()

    if unit in ("seconds", "minutes", "hours", "days", "weeks"):
        new_date = date + timedelta(**{unit: duration})
    elif unit in ("months", "years"):
        months = duration if unit == "months" else duration * 12
        total = date.month - 1 + months
        year, month = date.year + total // 12, total % 12 + 1
        # clamp the day to the last valid day of the target month
        next_month = datetime(year + (month == 12), month % 12 + 1, 1)
        last_day = (next_month - timedelta(days=1)).day
        new_date = date.replace(year=year, month=month, day=min(date.day, last_day))
    else:
        raise ValueError(f"Unsupported unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    """Stand-in for a real reminder system: just prints the reminder."""
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")
    return f"Reminder set for {timestamp}"

Quick check of the functions on their own, no API call:

In [ ]:
print(get_current_datetime("%H:%M:%S"))
print(add_duration_to_datetime("2050-01-01", 177, "days"))
print(add_duration_to_datetime("2026-01-31", 1, "months"))

## 4. Tool schemas

In [ ]:
get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format. "
                   "Use this whenever you need to know the current date or time.",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "Python strftime format string for the result.",
                "default": "%Y-%m-%d %H:%M:%S",
            }
        },
        "required": [],
    },
})

add_duration_to_datetime_schema = ToolParam({
    "name": "add_duration_to_datetime",
    "description": "Adds a duration to a date/time and returns the result as a formatted string "
                   "(e.g. 'Monday, June 27, 2050 12:00:00 AM'). Use this for any date arithmetic "
                   "instead of calculating dates yourself.",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The starting date/time, written to match input_format.",
            },
            "duration": {
                "type": "number",
                "description": "Amount of time to add. Use a negative number to subtract.",
            },
            "unit": {
                "type": "string",
                "enum": ["seconds", "minutes", "hours", "days", "weeks", "months", "years"],
                "description": "Unit of the duration.",
            },
            "input_format": {
                "type": "string",
                "description": "Python strftime format of datetime_str. Defaults to '%Y-%m-%d'.",
                "default": "%Y-%m-%d",
            },
        },
        "required": ["datetime_str", "duration", "unit"],
    },
})

set_reminder_schema = ToolParam({
    "name": "set_reminder",
    "description": "Sets a reminder that will notify the user at the given date and time.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "What the user should be reminded about.",
            },
            "timestamp": {
                "type": "string",
                "description": "When to send the reminder, in ISO 8601 format, e.g. '2050-06-27T09:00:00'.",
            },
        },
        "required": ["content", "timestamp"],
    },
})

TOOLS = [get_current_datetime_schema, add_duration_to_datetime_schema, set_reminder_schema]

## 5. Tool router and conversation loop

`run_tools` returns one `tool_result` block per `tool_use` block. If a tool raises an error, the error text goes back to Claude with `is_error: True` so it can recover or explain, instead of the notebook crashing.

In [ ]:
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    raise ValueError(f"Unknown tool: {tool_name}")


def run_tools(message):
    tool_results = []
    for block in message.content:
        if block.type != "tool_use":
            continue
        try:
            output = run_tool(block.name, block.input)
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": json.dumps(output),
                "is_error": False,
            })
        except Exception as e:
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": f"Error: {e}",
                "is_error": True,
            })
    return tool_results


def run_conversation(messages, tools=TOOLS, max_turns=10):
    """Keep calling Claude and running tools until it stops requesting them."""
    for _ in range(max_turns):
        response = chat(messages, tools=tools)
        add_assistant_message(messages, response)

        text = text_from_message(response)
        if text:
            print(f"Claude: {text}\n")

        if response.stop_reason != "tool_use":
            break

        add_user_message(messages, run_tools(response))
    return messages

## 6. Try it
### Single tool

In [ ]:
messages = []
add_user_message(messages, "What is the exact time, formatted as HH:MM:SS?")
run_conversation(messages)

### Multiple tools (the lesson 28 example)

In [ ]:
messages = []
add_user_message(messages, "Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050.")
run_conversation(messages)

### Inspect the full message history
Shows the user message, assistant text + tool_use blocks, tool_result messages, and the final reply.

In [ ]:
for m in messages:
    print(f"=== {m['role'].upper()} ===")
    content = m["content"]
    if isinstance(content, str):
        print(content)
    else:
        for block in content:
            print(block.model_dump() if hasattr(block, "model_dump") else block)
    print()

## Adding a new tool
1. Write the function (section 3)
2. Write its schema (section 4) and add it to `TOOLS`
3. Add an `elif` for it in `run_tool` (section 5)